In [1]:
import pandas as pd, numpy as np, json
from datetime import datetime
from pathlib import Path
from sklearn.cluster import KMeans

DATA_DIR = r"C:\Users\Prisha\Downloads\ml_turbofan\ml turbofan nasa\CMAPSSData"
OUT_DIR  = Path(r"C:\Users\Prisha\Downloads\ml_turbofan\data preprocessing") / "processed_output" / datetime.now().strftime("run_%Y%m%d_%H%M%S_%f")
SUBSETS  = ["FD001", "FD002", "FD003", "FD004"]
TRAIN_FRAC, A, B, SEED, W = 0.70, 60, 15, 42, 20       # W = rolling window (cycles)
DROP = ["s1", "s5", "s6", "s10", "s16", "s18", "s19"]  # near-constant sensors

op = ["op1", "op2", "op3"]
sensors = [f"s{i}" for i in range(1, 22)]
cols = ["unit", "cycle"] + op + sensors

# 1. load
frames = []
for s in SUBSETS:
    d = pd.read_csv(f"{DATA_DIR}/train_{s}.txt", sep=r"\s+", header=None, names=cols)
    d["subset"] = s
    d["unit"] = s + "_" + d["unit"].astype(str)
    frames.append(d)
df = pd.concat(frames, ignore_index=True)

# 2. quality checks
print("shape:", df.shape, "| missing:", int(df.isna().sum().sum()), "| duplicates:", int(df.duplicated().sum()))
df = df.dropna().drop_duplicates().reset_index(drop=True)

# 3. RUL and labels
df["rul"] = df.groupby("unit")["cycle"].transform("max") - df["cycle"]
df["label"] = np.select([df.rul <= B, df.rul <= A], ["anomaly", "near_anomaly"], "normal")

# 4. operating condition
df["cond"] = 0
for s in SUBSETS:
    m = df.subset == s
    k = 6 if s in ("FD002", "FD004") else 1
    if k > 1:
        df.loc[m, "cond"] = KMeans(k, n_init=10, random_state=SEED).fit_predict(df.loc[m, op])

# 5. split by engine, stratified by subset
rng = np.random.default_rng(SEED)
train_units = []
for s in SUBSETS:
    u = df.loc[df.subset == s, "unit"].unique()
    train_units += list(rng.choice(u, int(len(u) * TRAIN_FRAC), replace=False))
df["split"] = np.where(df.unit.isin(train_units), "train", "test")

# 6. sensor std table (review it)
ref = df[(df.split == "train") & (df.label == "normal")]
print(ref.groupby("subset")[sensors].std().round(4).T)

# 7. standardize per (subset, condition) using training healthy rows only
feat = [s for s in sensors if s not in DROP]
mu = ref.groupby(["subset", "cond"])[feat].mean()
sd = ref.groupby(["subset", "cond"])[feat].std().replace(0, 1)
idx = pd.MultiIndex.from_frame(df[["subset", "cond"]])
df[feat] = (df[feat].values - mu.reindex(idx).values) / sd.reindex(idx).values
assert not df[feat].isna().any().any()

# 8. NEW (v2): rolling-trend features, past cycles only, per engine
df = df.sort_values(["unit", "cycle"]).reset_index(drop=True)
roll = df.groupby("unit")[feat].rolling(W, min_periods=1).mean().reset_index(level=0, drop=True)
roll.columns = [c + "_m" for c in feat]
df = pd.concat([df, roll], axis=1)
feat = feat + list(roll.columns)          # 28 features

# 9. save
meta = ["unit", "subset", "cycle", "rul", "label", "split"]
OUT_DIR.mkdir(parents=True, exist_ok=False)
df[meta + feat].to_csv(OUT_DIR / "turbofan_full.csv", index=False)
df[(df.split == "train") & (df.label == "normal")][meta + feat].to_csv(OUT_DIR / "train_normal.csv", index=False)
df[df.split == "test"][meta + feat].to_csv(OUT_DIR / "test_all.csv", index=False)
with (OUT_DIR / "features.json").open("w", encoding="utf-8") as f:
    json.dump(feat, f)
print("Saved to:", OUT_DIR)
print(len(feat), "features")
print(df.groupby(["split", "label"]).size())

shape: (160359, 27) | missing: 0 | duplicates: 0
subset    FD001     FD002    FD003     FD004
s1       0.0000   26.3486   0.0000   26.4281
s2       0.3847   37.2356   0.4100   37.3586
s3       4.7906  105.7477   5.3727  106.1668
s4       6.3045  118.6220   6.9394  118.9953
s5       0.0000    3.6091   0.0000    3.6210
s6       0.0015    5.4252   0.0177    5.4424
s7       0.6407  145.9722   1.9628  146.7416
s8       0.0548  145.4349   0.1448  145.9898
s9      10.0005  335.3634  10.7813  336.8075
s10      0.0000    0.1276   0.0017    0.1277
s11      0.1827    3.2219   0.2038    3.2391
s12      0.5195  137.6203   1.8505  138.3399
s13      0.0554  128.3626   0.1444  128.8441
s14      9.1807   84.0822   8.9595   84.6223
s15      0.0273    0.7495   0.0393    0.7512
s16      0.0000    0.0046   0.0000    0.0046
s17      1.1819   27.6936   1.3508   27.7980
s18      0.0000  145.5530   0.0000  146.1172
s19      0.0000    5.3763   0.0000    5.3966
s20      0.1340    9.8789   0.1707    9.9382
s21   

In [2]:
df.head()

,unit,cycle,op1,op2,op3,s1,s2,s3,s4,s5,...,s8_m,s9_m,s11_m,s12_m,s13_m,s14_m,s15_m,s17_m,s20_m,s21_m
0,FD001_1,1,-0.0007,-0.0004,100.0,518.67,-1.777185,0.276433,-0.755260,14.62,...,-0.255448,-1.341493,0.197901,-0.078524,-0.967894,-0.124514,-0.312705,-0.549363,1.318098,1.107999
1,FD001_1,2,0.0019,-0.0003,100.0,518.67,-0.919269,0.718970,-0.352370,14.62,...,-0.437856,-1.447488,0.252632,0.518186,-0.516872,-0.512829,-0.087554,-0.549363,1.094194,1.136399
2,FD001_1,3,-0.0043,0.0003,100.0,518.67,-0.399320,-0.080518,-0.184236,14.62,...,-0.255448,-1.187167,-0.130485,0.806916,-0.607076,-0.579091,-0.183350,-1.113429,0.895169,0.819061
3,FD001_1,4,0.0007,0.0000,100.0,518.67,-0.399320,-1.165984,-0.553815,14.62,...,-0.027439,-1.143502,-0.513601,1.163017,-0.426667,-0.595883,-0.685212,-0.972412,0.665045,0.752075
4,FD001_1,5,-0.0019,-0.0002,100.0,518.67,-0.347325,-1.153459,0.136172,14.62,...,-0.073041,-1.003908,-0.579278,1.118745,-0.462749,-0.606612,-0.538223,-0.718583,0.556825,0.787204
